## Incremental Bronze-to-Silver Processing — Design Decisions

This section records the main design decisions behind the Bronze-to-Silver process so the reasoning is not lost later.

### 1. Stores and Products — Full Snapshot

Stores and Products are small master datasets, so every run reads the full source and replaces the Silver tables.

Why:
- The datasets are small.
- No historical attribute tracking is required.
- Full replacement is simpler than incremental change detection.

Chosen approach:

`Source Snapshot → Replace Silver Table`

We intentionally did not add watermark or MERGE logic because it would add complexity without meaningful benefit for these small tables.

---

### 2. Initial Load Before Incremental Processing

The initial load must run first.

It creates:
- The January Silver baseline.
- The pipeline control table.
- The initial pipeline watermark.

The incremental notebook checks that these objects exist before continuing.

Why:

Incremental processing needs a known starting point. Without the initial baseline, the pipeline would not know what has already been processed.

---

### 3. Persistent Pipeline Watermark

The last successfully processed point is stored in:

`control.pipeline_state`

rather than only in a Python variable.

Why:

Python variables exist only during the current notebook session. The control table persists between pipeline runs.

The watermark acts like a bookmark showing where the previous successful run stopped.

---

### 4. Fixed Incremental Header Window

The Mirrored Database keeps changing while the notebook is running.

At the beginning of each run, we capture:

`source_high_watermark = MAX(LastModifiedAt)`

Then process:

`pipeline_watermark < LastModifiedAt <= source_high_watermark`

Why:

This creates a fixed processing window even though Mirroring continues receiving new data.

Anything arriving after the captured high watermark belongs to the next run.

---

### 5. Why Headers Use `LastModifiedAt`

Headers can already exist in Silver and later change, such as an invoice status moving from Pending to Completed.

Therefore:

`LastModifiedAt`

is used for incremental change detection.

`InvoiceDate` cannot be used for this because it represents when the transaction originally happened, not when it was later changed.

The current header batch can therefore contain both:

`New Headers + Previously Loaded Headers Updated Later`

---

### 6. Header Business Validation Uses All Source Lines

A changed header is considered valid when at least one valid line exists for its `InvoiceID`.

The check uses all source lines, not only new lines.

Why:

A January header may be updated in February while its invoice lines were already processed in January.

Checking only new lines could incorrectly quarantine that header.

Rule:

`At least one line with Quantity IS NOT NULL → Eligible Header`

`No valid lines → Quarantine Header`

---

### 7. Detecting New Invoice Lines

Invoice Lines are treated as immutable and do not have `LastModifiedAt`.

Therefore, new lines are detected using `LineID`.

A line is processed only when:

1. Its `LineID` does not already exist in either the clean or quarantine Silver tables.
2. Its `InvoiceID` belongs to the current changed-header batch.

Why both conditions are needed:

The first prevents duplicate processing of old lines.

The second prevents continuously mirrored future lines from entering the current watermark window.

---

### 8. Line Data Quality and Write Strategy

After detecting new lines:

`Quantity IS NOT NULL → silver.invoices_lines`

`Quantity IS NULL → silver.quarantine_invoices_lines`

Lines use `APPEND` because they are treated as immutable.

Previously processed LineIDs are excluded before writing, making repeated processing safe.

---

### 9. Eligible Header MERGE

Eligible headers use `MERGE` on `InvoiceID`.

`MATCHED → UPDATE`

`NOT MATCHED → INSERT`

Why:

The same header may already exist from an earlier load but contain a newer status or `LastModifiedAt`.

A simple append would create duplicates and would not correctly handle updates.

---

### 10. Quarantine Header MERGE

Quarantine headers also use `MERGE`, but only:

`NOT MATCHED → INSERT`

No update is currently required.

Why:

If the same batch is retried, appending quarantine headers would create duplicate quarantine records.

The MERGE makes this step idempotent.

---

### 11. Watermark Advancement

The pipeline watermark is updated only after the incremental processing succeeds.

The new watermark becomes the captured `source_high_watermark`.

Why:

The watermark represents successfully processed data.

If processing fails before completion, the watermark must remain unchanged so the same batch can be processed again safely.

---

## Final Processing Pattern

`Stores / Products → Full Snapshot`

`Headers → Watermark Window → Validate → MERGE / Quarantine`

`Lines → Detect New LineIDs → Validate → Append / Quarantine`

`Successful Processing → Advance Watermark`


In [1]:
# Products Processing
    # Reading data frames
df_products = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("Files/bronze/stores_products_data/Products_Master.csv")
    # View creation
df_products.createOrReplaceTempView("vw_products")
    # Apply transformation logic and save clean data
spark.sql(
    """
        CREATE OR REPLACE TABLE silver.products AS
        SELECT
            ProductID,
            ProductName,
            Category,
            CAST(UnitPrice AS DECIMAL(10,2)) AS UnitPrice
        FROM vw_products
    """
        )

StatementMeta(, 06722686-dc40-4630-8acd-f8a89b40337b, 3, Finished, Available, Finished, False)

DataFrame[]

In [2]:
# Stores Processing
    # Reading data frames
df_stores = spark.read.format("csv").option("header","true").option("inferSchema", "true").load("Files/bronze/stores_products_data/Stores_Catalog.csv")
    # View creation
df_stores.createOrReplaceTempView("vw_stores")
    # Apply transformation logic and save clean data
spark.sql("""
    CREATE OR REPLACE TABLE silver.stores AS
    SELECT
        StoreID,
        StoreName,
        Region,
        City
    FROM vw_stores
""")

StatementMeta(, 06722686-dc40-4630-8acd-f8a89b40337b, 4, Finished, Available, Finished, False)

DataFrame[]

In [3]:
# Invoices line processing + Preparing execution plan for header processing

from pyspark.sql import functions as f

# Prerequisite: nb_initial_load must run before this incremental pipeline
if not spark.catalog.tableExists("silver.invoices_header"):
    raise Exception("Initial Load must run first.")

if not spark.catalog.tableExists("control.pipeline_state"):
    raise Exception("Pipeline control state is missing. Run Initial Load first.")

# Read the last successful pipeline checkpoint
pipeline_watermark = spark.sql("""
    SELECT PipelineWatermark
    FROM control.pipeline_state
    WHERE PipelineName = 'bronze_to_silver'
""").collect()[0]["PipelineWatermark"]

# Determine source high watermark for the current incremental run
source_high_watermark = (
    spark.read.format("delta")
    .load("Files/bronze/pos_retail_transactions/Invoices_Header")
    .agg(f.max("LastModifiedAt").alias("source_high_watermark"))
    .collect()[0]["source_high_watermark"]
)

# Read bronze with watermark
df_changed_headers = (
    spark.read.format("delta")
    .load("Files/bronze/pos_retail_transactions/Invoices_Header")
    .filter(
        (f.col("LastModifiedAt") > pipeline_watermark) &
        (f.col("LastModifiedAt") <= source_high_watermark)
    )
)

df_all_source_lines = (
    spark.read.format("delta")
    .load("Files/bronze/pos_retail_transactions/Invoices_Lines")
)

df_all_source_lines.createOrReplaceTempView('vw_all_source_lines')
df_changed_headers.createOrReplaceTempView('vw_changed_headers')


df_eligible_headers = spark.sql("""
            SELECT * FROM  vw_changed_headers AS h 
            WHERE EXISTS (SELECT 1 FROM vw_all_source_lines AS l where h.InvoiceID = l.InvoiceID and l.Quantity IS NOT NULL)
                                """)

df_quarantine_headers = spark.sql("""
            SELECT * FROM  vw_changed_headers AS h 
            WHERE NOT EXISTS (SELECT 1 FROM vw_all_source_lines AS l where h.InvoiceID = l.InvoiceID and l.Quantity IS NOT NULL)
                                """).\
                                withColumn("QuarantineReason", f.lit("Header without valid line items")).withColumn("QuarantinedAt", f.current_timestamp())

# Exclude already processed LineIDs and keep only lines linked to the current changed-header batch
df_new_lines = spark.sql("""
            WITH t as (
                SELECT LineID FROM silver.invoices_lines UNION ALL SELECT LineID FROM silver.quarantine_invoices_lines
            )
            SELECT * FROM vw_all_source_lines AS l WHERE NOT EXISTS(SELECT 1 FROM t WHERE l.LineID = t.LineID)
                                                        AND EXISTS (SELECT 1 FROM vw_changed_headers as h WHERE l.InvoiceID = h.InvoiceId) 
            """)
df_new_lines.createOrReplaceTempView("vw_new_lines")

df_quarantine_lines = spark.sql(""" SELECT * FROM  vw_new_lines WHERE Quantity IS NULL """).\
    withColumn("QuarantineReason", f.lit("Missing Quantity")).withColumn("QuarantinedAt", f.current_timestamp())

df_eligible_lines = spark.sql(""" SELECT * FROM  vw_new_lines WHERE Quantity IS NOT NULL """)

print(f"Count of new lines before incremental loading: {df_new_lines.count()}")
print(f"Count of new valid lines before incremental loading: {df_eligible_lines.count()}")
print(f"Count of new not valid lines before incremental loading: {df_quarantine_lines.count()}")

df_eligible_lines.write.format("Delta").mode("append").saveAsTable("silver.invoices_lines")
df_quarantine_lines.write.format("Delta").mode("append").saveAsTable("silver.quarantine_invoices_lines")

print(f"Count of unprocessed lines after incremental loading: {df_new_lines.count()}")
print(f"Count of unprocessed valid lines after incremental loading: {df_eligible_lines.count()}")
print(f"Count of unprocessed not valid lines after incremental loading: {df_quarantine_lines.count()}")

StatementMeta(, 06722686-dc40-4630-8acd-f8a89b40337b, 5, Finished, Available, Finished, False)

Count of new lines before incremental loading: 553238
Count of new valid lines before incremental loading: 525718
Count of new not valid lines before incremental loading: 27520
Count of unprocessed lines after incremental loading: 0
Count of unprocessed valid lines after incremental loading: 0
Count of unprocessed not valid lines after incremental loading: 0


In [9]:
# Invoices header processing 

df_eligible_headers.createOrReplaceTempView("vw_silver_invoices_header")
df_quarantine_headers.createOrReplaceTempView("vw_silver_quarantine_header")
# Upsert data 
spark.sql(
    """
        MERGE INTO silver.invoices_header AS target
        USING vw_silver_invoices_header  AS source
        ON target.InvoiceID = source.InvoiceID
        WHEN MATCHED THEN
            UPDATE SET
                target.OrderStatus = source.OrderStatus,
                target.LastModifiedAt = source.LastModifiedAt
        WHEN NOT MATCHED THEN
            INSERT (InvoiceID, StoreID, InvoiceDate, PaymentMethod, OrderStatus, LastModifiedAt)
            VALUES (source.InvoiceID, source.StoreID, source.InvoiceDate, source.PaymentMethod, source.OrderStatus, source.LastModifiedAt);

    """
)

 
spark.sql(
    """
        MERGE INTO silver.quarantine_invoices_header AS target
        USING vw_silver_quarantine_header AS source
        ON target.InvoiceID = source.InvoiceID
        WHEN NOT MATCHED THEN
            INSERT (InvoiceID, StoreID, InvoiceDate, PaymentMethod, OrderStatus, LastModifiedAt, QuarantineReason, QuarantinedAt)
            VALUES (source.InvoiceID, source.StoreID, source.InvoiceDate, source.PaymentMethod, source.OrderStatus, source.LastModifiedAt, source.QuarantineReason, source.QuarantinedAt)

    """
)   

spark.sql(f"""
    UPDATE control.pipeline_state
    SET
        PipelineWatermark = TIMESTAMP('{source_high_watermark}'),
        UpdatedAt = current_timestamp()
    WHERE PipelineName = 'bronze_to_silver'
""")

StatementMeta(, 06722686-dc40-4630-8acd-f8a89b40337b, 11, Finished, Available, Finished, False)

DataFrame[num_affected_rows: bigint]

In [13]:
# Check pipeline and source high watermark
updated_pipeline_watermark = spark.sql("""
    SELECT PipelineWatermark
    FROM control.pipeline_state
    WHERE PipelineName = 'bronze_to_silver'
""").collect()[0]["PipelineWatermark"]

print("Updated pipeline watermark:", updated_pipeline_watermark)
print("Source high watermark:", source_high_watermark)

StatementMeta(, 06722686-dc40-4630-8acd-f8a89b40337b, 15, Finished, Available, Finished, False)

Updated pipeline watermark: 2026-02-28 23:59:59
Source high watermark: 2026-02-28 23:59:59


In [14]:
%%sql
-- Total clean headers
SELECT COUNT(*) FROM silver.invoices_header;

-- Total quarantine headers
SELECT COUNT(*) FROM silver.quarantine_invoices_header;

StatementMeta(, 06722686-dc40-4630-8acd-f8a89b40337b, 17, Finished, Available, Finished, True)

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1 rows and 1 fields>

In [1]:
%%sql
-- Check for clean invoice lines without a matching clean header
SELECT COUNT(*)
FROM silver.invoices_lines AS l
LEFT JOIN silver.invoices_header AS h
    ON l.InvoiceID = h.InvoiceID
WHERE h.InvoiceID IS NULL;

StatementMeta(, 33fb9c9b-14cc-4202-85c6-3ae4f07b730d, 2, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [17]:
%%sql
-- Showing the total count of headers Jan. + Feb for clean and quarantine
SELECT
    'Count clean header' AS count_type,
    COUNT(*) AS total_count
FROM silver.invoices_header

UNION ALL

SELECT
    'Count quarantine header',
    COUNT(*)
FROM silver.quarantine_invoices_header

StatementMeta(, 06722686-dc40-4630-8acd-f8a89b40337b, 20, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 2 fields>

In [15]:
# Total number of new header candidates --> Feb. headers + Jan. Updated headers
print(f"Header candidates: {df_changed_headers.count()}")

StatementMeta(, 06722686-dc40-4630-8acd-f8a89b40337b, 18, Finished, Available, Finished, False)

Header candidates: 277943


In [16]:
# Showing the headers seperated into eligible and quarantine
print(f"Eligible headers: {df_eligible_headers.count()}")
print(f"Quarantine headers: {df_quarantine_headers.count()}")

StatementMeta(, 06722686-dc40-4630-8acd-f8a89b40337b, 19, Finished, Available, Finished, False)

Eligible headers: 273196
Quarantine headers: 4747
